In [31]:
import pandas as pd
import numpy as np
from pydantic import BaseModel, Field, field_validator, computed_field, model_validator, ConfigDict, ValidationError
from typing import Annotated, Optional, List, Dict, Literal
from enum import IntEnum
from datetime import datetime

In [32]:
FILE_PATH = "data_predictive_maintenance.csv"
df = pd.read_csv(FILE_PATH)

In [33]:
df.head()

,timestamp,machine_id,temperature,pressure,vibration,rpm,voltage,current,humidity,operating_hours,maintenance_days,fault_code,failure_next_7_days,production_batch
0,1/1/2025 0:00,Machine_C,NaN,33.55,2.19,2046.32,222.67,10.81,34.89,1019,24,0,1,B1000
1,1/1/2025 6:00,Machine_D,67.04,30.59,2.22,1716.58,226.27,11.36,35.87,1016,49,0,0,B1000
2,1/1/2025 12:00,Machine_D,64.55,27.76,2.59,1759.36,230.54,8.86,49.56,1010,21,0,1,B1000
3,1/1/2025 18:00,Machine_A,70.67,30.20,1.88,1691.69,244.82,10.98,44.42,1050,78,0,0,B1000
4,1/2/2025 0:00,Machine_D,71.25,26.12,1.47,1835.44,235.91,11.26,53.84,1043,82,0,0,B1000


In [34]:
df.shape

(200, 14)

In [35]:
df.duplicated().sum()

np.int64(0)

In [36]:
df.isnull().sum()

timestamp               0
machine_id              0
temperature            16
pressure               10
vibration               0
rpm                     0
voltage                20
current                 0
humidity                0
operating_hours         0
maintenance_days        0
fault_code              0
failure_next_7_days     0
production_batch        0
dtype: int64

In [39]:
df.replace([r'^\s*$', "N/A", np.nan, "NULL", "null"], pd.NA, regex=True, inplace=True)
df.head(5)

,timestamp,machine_id,temperature,pressure,vibration,rpm,voltage,current,humidity,operating_hours,maintenance_days,fault_code,failure_next_7_days,production_batch
0,1/1/2025 0:00,Machine_C,<NA>,33.55,2.19,2046.32,222.67,10.81,34.89,1019,24,0,1,B1000
1,1/1/2025 6:00,Machine_D,67.04,30.59,2.22,1716.58,226.27,11.36,35.87,1016,49,0,0,B1000
2,1/1/2025 12:00,Machine_D,64.55,27.76,2.59,1759.36,230.54,8.86,49.56,1010,21,0,1,B1000
3,1/1/2025 18:00,Machine_A,70.67,30.2,1.88,1691.69,244.82,10.98,44.42,1050,78,0,0,B1000
4,1/2/2025 0:00,Machine_D,71.25,26.12,1.47,1835.44,235.91,11.26,53.84,1043,82,0,0,B1000


In [40]:
print(df.isnull().sum())

timestamp               0
machine_id              0
temperature            16
pressure               10
vibration               0
rpm                     0
voltage                20
current                 0
humidity                0
operating_hours         0
maintenance_days        0
fault_code              0
failure_next_7_days     0
production_batch        0
dtype: int64


In [42]:
df.isnull().sum().value_counts()

0     11
16     1
10     1
20     1
Name: count, dtype: int64

In [43]:
df.dtypes

timestamp                  str
machine_id                 str
temperature             object
pressure                object
vibration              float64
rpm                    float64
voltage                 object
current                float64
humidity               float64
operating_hours          int64
maintenance_days         int64
fault_code               int64
failure_next_7_days      int64
production_batch           str
dtype: object

In [44]:
df.columns

Index(['timestamp', 'machine_id', 'temperature', 'pressure', 'vibration',
       'rpm', 'voltage', 'current', 'humidity', 'operating_hours',
       'maintenance_days', 'fault_code', 'failure_next_7_days',
       'production_batch'],
      dtype='str')

In [47]:
# Correcting datatypes

# errors='coerce': If a value cannot be parsed as a date (e.g., 'invalid_date', 'abc', or bad formatting), it replaces that specific value with NaT
df.timestamp = pd.to_datetime(df.timestamp, errors='coerce')

FEATURE_COLUMNS = ['temperature', 'pressure', 'vibration',
       'rpm', 'voltage', 'current', 'humidity', 'operating_hours',
       'maintenance_days']


'''
In pd.to_numeric(..., errors='coerce'), errors='coerce' forces any non-numeric values (such as text strings, symbols, or corrupt data)
to become missing numeric values (NaN).If a column contains values like 'invalid', 'N/A', or '35.2%', pandas will not raise an error;
instead, it converts those specific bad entries into NaN while successfully parsing valid numbers (like '25.4' $\rightarrow$ 25.4).
'''
for col in FEATURE_COLUMNS:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(df.dtypes)

timestamp              datetime64[us]
machine_id                        str
temperature                   float64
pressure                      float64
vibration                     float64
rpm                           float64
voltage                       float64
current                       float64
humidity                      float64
operating_hours                 int64
maintenance_days                int64
fault_code                      int64
failure_next_7_days             int64
production_batch                  str
dtype: object


In [48]:
df.head()

,timestamp,machine_id,temperature,pressure,vibration,rpm,voltage,current,humidity,operating_hours,maintenance_days,fault_code,failure_next_7_days,production_batch
0,2025-01-01 00:00:00,Machine_C,NaN,33.55,2.19,2046.32,222.67,10.81,34.89,1019,24,0,1,B1000
1,2025-01-01 06:00:00,Machine_D,67.04,30.59,2.22,1716.58,226.27,11.36,35.87,1016,49,0,0,B1000
2,2025-01-01 12:00:00,Machine_D,64.55,27.76,2.59,1759.36,230.54,8.86,49.56,1010,21,0,1,B1000
3,2025-01-01 18:00:00,Machine_A,70.67,30.20,1.88,1691.69,244.82,10.98,44.42,1050,78,0,0,B1000
4,2025-01-02 00:00:00,Machine_D,71.25,26.12,1.47,1835.44,235.91,11.26,53.84,1043,82,0,0,B1000


In [51]:
df.fault_code.value_counts()

fault_code
0    124
1     40
2     26
3     10
Name: count, dtype: int64

In [56]:
class FaultCode(IntEnum):
    No_Fault = 0
    Bearing_Failure = 1
    Motor_Failure = 2
    Pump_Failure = 3

In [53]:
FaultCode.Pump_Failure.value

3

In [57]:
f = FaultCode(1)

In [58]:
print(f)

1


In [59]:
print(f.name)

Bearing_Failure


In [62]:
class SensorData(BaseModel):
    model_config = ConfigDict(extra='forbid', validate_assignment=True)

    temperature : Annotated[float, Field(gt=10,  le=102)]
    pressure    : Annotated[float, Field(gt=20,  le=50)]
    vibration   : Annotated[float, Field(gt=0,   le=5)]
    rpm         : Annotated[float, Field(gt=0,   le=20000)]
    voltage     : Annotated[float, Field(gt=220, le=260)]
    current     : Annotated[float, Field(gt=5,   le=20)]
    humidity    : Annotated[float, Field(gt=25,  le=100)]

    sensor_metadata: Optional[Dict[str, str]] = None

    @computed_field
    @property
    def power_consumption(self) -> float:
        return round(self.voltage * self.current, 2)

    @computed_field
    @property
    def vibration_status(self) -> str:
        if self.vibration > 5:
            return "Critical"
        elif self.vibration > 2:
            return "Warning"
        return "Normal"


'''
SensorData : Allow an optional dictionary containing additional sensor information; if no information is provided, use None.
│
├── Measurements
│   ├── temperature
│   ├── pressure
│   ├── vibration
│   ├── rpm
│   ├── voltage
│   ├── current
│   └── humidity
│
└── Metadata (optional)
    ├── sensor_id
    ├── location
    └── manufacturer
'''

'\nSensorData : Allow an optional dictionary containing additional sensor information; if no information is provided, use None.\n│\n├── Measurements\n│   ├── temperature\n│   ├── pressure\n│   ├── vibration\n│   ├── rpm\n│   ├── voltage\n│   ├── current\n│   └── humidity\n│\n└── Metadata (optional)\n    ├── sensor_id\n    ├── location\n    └── manufacturer\n'

In [63]:
class MachineInfo(BaseModel):
    machine_id: str
    production_batch: str
    
    tages: Optional[List[str]] = None
    metadata: Optional[Dict[str, str]] = None

    @field_validator("machine_id")
    @classmethod
    def validate_machine_id(cls, value):
        if not value.startswith("Machine_"):
            raise ValueError("Invalid Machine ID")
        return value

In [64]:
# Maintenance Model

class MaintenanceInfo(BaseModel):
    operating_hours: int
    maintenance_days: int
    fault_code: FaultCode

In [65]:
# Prediction Model

class PredictionInfo(BaseModel):

    failure_next_7_days: Literal[0, 1]

    # @field_validator('failure_next_7_days')
    # @classmethod
    # def validate_target(cls, value):
    #     if value not in [0, 1]:
    #         raise ValueError("Target must be 0 or 1")
    #     return value

In [67]:
df.dtypes

timestamp              datetime64[us]
machine_id                        str
temperature                   float64
pressure                      float64
vibration                     float64
rpm                           float64
voltage                       float64
current                       float64
humidity                      float64
operating_hours                 int64
maintenance_days                int64
fault_code                      int64
failure_next_7_days             int64
production_batch                  str
dtype: object

In [68]:
# Root nested model

class MachineReading(BaseModel):

    timestamp: datetime
    machine: MachineInfo
    sensors: SensorData
    maintenance: MaintenanceInfo
    prediction: PredictionInfo

    @model_validator(mode="after")
    def validate_machine_state(self):
        if (
            self.sensors.temperature > 100
            and
            self.sensors.pressure < 10
        ):
            raise ValueError("Abnormal operating condition")
        return self

    @computed_field
    @property
    def health_score(self) -> float:
        score = 100.0

        # Vibration penalty: maximum 50 points
        score -= min(self.sensors.vibration * 3, 50)

        # Operating-hours penalty: maximum 30 points
        score -= min(self.maintenance.operating_hours / 500, 30)

        # Keep score between 0 and 100
        return round(max(score, 0), 2)

In [69]:
def row_to_dict(row: pd.Series) -> dict:
    """
    Converts a flat DataFrame row into the nested dict structure
    that MachineReading.model_validate() expects.
    """

    def safe(val):
        try:
            if pd.isna(val):
                return None
        except (TypeError, ValueError):
            pass
        if isinstance(val, np.integer):
            return int(val)
        if isinstance(val, np.floating):
            return float(val)
        return val

    return {
        "timestamp": safe(row["timestamp"]),
        "machine": {
            "machine_id"       : safe(row["machine_id"]),
            "production_batch" : safe(row["production_batch"]),
        },
        "sensors": {
            "temperature" : safe(row["temperature"]),
            "pressure"    : safe(row["pressure"]),
            "vibration"   : safe(row["vibration"]),
            "rpm"         : safe(row["rpm"]),
            "voltage"     : safe(row["voltage"]),
            "current"     : safe(row["current"]),
            "humidity"    : safe(row["humidity"]),
        },
        "maintenance": {
            "operating_hours"  : safe(row["operating_hours"]),
            "maintenance_days" : safe(row["maintenance_days"]),
            "fault_code"       : safe(row["fault_code"]),
        },
        "prediction": {
            "failure_next_7_days": safe(row["failure_next_7_days"]),
        },
    }

# Quick smoke test — run this to confirm the function works before Stage 2
sample = row_to_dict(df.iloc[0])
print(sample)

{'timestamp': Timestamp('2025-01-01 00:00:00'), 'machine': {'machine_id': 'Machine_C', 'production_batch': 'B1000'}, 'sensors': {'temperature': None, 'pressure': 33.55, 'vibration': 2.19, 'rpm': 2046.32, 'voltage': 222.67, 'current': 10.81, 'humidity': 34.89}, 'maintenance': {'operating_hours': 1019, 'maintenance_days': 24, 'fault_code': 0}, 'prediction': {'failure_next_7_days': 1}}


In [72]:
for idx, row in df.iterrows():
    print(idx, row)

0 timestamp              2025-01-01 00:00:00
machine_id                       Machine_C
temperature                            NaN
pressure                             33.55
vibration                             2.19
rpm                                2046.32
voltage                             222.67
current                              10.81
humidity                             34.89
operating_hours                       1019
maintenance_days                        24
fault_code                               0
failure_next_7_days                      1
production_batch                     B1000
Name: 0, dtype: object
1 timestamp              2025-01-01 06:00:00
machine_id                       Machine_D
temperature                          67.04
pressure                             30.59
vibration                             2.22
rpm                                1716.58
voltage                             226.27
current                              11.36
humidity                   

In [73]:

valid_records   = []   # dicts ready for MongoDB
invalid_records = []   # original row + error details

for idx, row in df.iterrows():
    raw = row_to_dict(row)           # Stage 1 output feeds directly here
    try:
        reading = MachineReading.model_validate(raw)

        doc = reading.model_dump(mode="json")   # datetime + IntEnum → JSON-safe types
        doc["_source_row"] = int(idx)           # audit trail: original CSV row number
        valid_records.append(doc)

    except ValidationError as e:
        errors = [
            f"{' -> '.join(str(loc) for loc in err['loc'])}: {err['msg']}"
            for err in e.errors()
        ]
        bad_row = row.to_dict()
        bad_row["_source_row"]       = int(idx)
        bad_row["validation_errors"] = " | ".join(errors)
        invalid_records.append(bad_row)

print(f"✅ Valid rows   : {len(valid_records)}")
print(f"❌ Invalid rows : {len(invalid_records)}")

# Preview first invalid row's errors if any
if invalid_records:
    print(f"\nSample error (row {invalid_records[0]['_source_row']}):")
    print(invalid_records[0]["validation_errors"])

✅ Valid rows   : 157
❌ Invalid rows : 43

Sample error (row 0):
sensors -> temperature: Input should be a valid number


In [74]:
valid_df = pd.json_normalize(valid_records)
print(valid_df.shape)
valid_df.head()

(157, 21)


,timestamp,health_score,_source_row,machine.machine_id,machine.production_batch,machine.tages,machine.metadata,sensors.temperature,sensors.pressure,sensors.vibration,...,sensors.voltage,sensors.current,sensors.humidity,sensors.sensor_metadata,sensors.power_consumption,sensors.vibration_status,maintenance.operating_hours,maintenance.maintenance_days,maintenance.fault_code,prediction.failure_next_7_days
0,2025-01-01T06:00:00,91.31,1,Machine_D,B1000,None,None,67.04,30.59,2.22,...,226.27,11.36,35.87,None,2570.43,Warning,1016,49,0,0
1,2025-01-01T12:00:00,90.21,2,Machine_D,B1000,None,None,64.55,27.76,2.59,...,230.54,8.86,49.56,None,2042.58,Warning,1010,21,0,1
2,2025-01-01T18:00:00,92.26,3,Machine_A,B1000,None,None,70.67,30.20,1.88,...,244.82,10.98,44.42,None,2688.12,Normal,1050,78,0,0
3,2025-01-02T00:00:00,93.50,4,Machine_D,B1000,None,None,71.25,26.12,1.47,...,235.91,11.26,53.84,None,2656.35,Normal,1043,82,0,0
4,2025-01-02T06:00:00,94.00,5,Machine_A,B1000,None,None,68.19,33.03,1.29,...,226.92,9.98,61.12,None,2264.66,Normal,1063,63,1,0


In [75]:
valid_df.columns = valid_df.columns.str.replace(
    r'^(machine|sensors|maintenance|prediction)\.', '', regex=True
)
print(valid_df.columns.tolist())

['timestamp', 'health_score', '_source_row', 'machine_id', 'production_batch', 'tages', 'metadata', 'temperature', 'pressure', 'vibration', 'rpm', 'voltage', 'current', 'humidity', 'sensor_metadata', 'power_consumption', 'vibration_status', 'operating_hours', 'maintenance_days', 'fault_code', 'failure_next_7_days']


In [76]:
valid_df.head()

,timestamp,health_score,_source_row,machine_id,production_batch,tages,metadata,temperature,pressure,vibration,...,voltage,current,humidity,sensor_metadata,power_consumption,vibration_status,operating_hours,maintenance_days,fault_code,failure_next_7_days
0,2025-01-01T06:00:00,91.31,1,Machine_D,B1000,None,None,67.04,30.59,2.22,...,226.27,11.36,35.87,None,2570.43,Warning,1016,49,0,0
1,2025-01-01T12:00:00,90.21,2,Machine_D,B1000,None,None,64.55,27.76,2.59,...,230.54,8.86,49.56,None,2042.58,Warning,1010,21,0,1
2,2025-01-01T18:00:00,92.26,3,Machine_A,B1000,None,None,70.67,30.20,1.88,...,244.82,10.98,44.42,None,2688.12,Normal,1050,78,0,0
3,2025-01-02T00:00:00,93.50,4,Machine_D,B1000,None,None,71.25,26.12,1.47,...,235.91,11.26,53.84,None,2656.35,Normal,1043,82,0,0
4,2025-01-02T06:00:00,94.00,5,Machine_A,B1000,None,None,68.19,33.03,1.29,...,226.92,9.98,61.12,None,2264.66,Normal,1063,63,1,0


In [77]:
import os
from dotenv import load_dotenv

from pymongo import MongoClient, InsertOne
from pymongo.errors import BulkWriteError


# Load variables from .env
load_dotenv()

MONGO_URI = os.getenv("MONGO_URI")

DB_NAME    = "predictive_maintenance"
COLLECTION_NAME = "machine_sensor_readings_01"

if valid_records:
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)

    try:
        client.admin.command("ping")
        print("Connected to MongoDB\n")

        coll       = client[DB_NAME][COLLECTION_NAME]
        operations = [InsertOne(doc) for doc in valid_records]

        result = coll.bulk_write(operations, ordered=False)
        print(f"Inserted : {result.inserted_count} documents")
        print(f"Collection : {DB_NAME}.{COLLECTION_NAME}")

    except ConnectionError:
        print("Could not connect to MongoDB. Is it running?")

    except BulkWriteError as bwe:
        inserted = bwe.details.get("nInserted", 0)
        errors   = bwe.details.get("writeErrors", [])
        print(f"Partial insert — {inserted} succeeded, {len(errors)} failed")
        for err in errors:
            print(f"   Doc index {err['index']}: {err['errmsg']}")

    finally:
        client.close()
        print("Connection closed")

else:
    print("⚠️ No valid records to insert.")

Connected to MongoDB

Inserted : 157 documents
Collection : predictive_maintenance.machine_sensor_readings_01
Connection closed
